# P1 — Severity-Stratified LoRA Adapters on Frozen B2

**Architecture:**
- WavLM-base backbone: loaded from B2 checkpoint, fully frozen
- 4 LoRA adapter sets (rank=8) injected into q_proj + v_proj of all 12 transformer layers
  - LoRA_control, LoRA_mild, LoRA_moderate, LoRA_severe
  - Each adapter trained only on its severity class utterances
- Severity classifier: 2-layer MLP (768 → 128 → 4) on mean-pooled layer-6 output
- CTC head: initialised from B2 weights, fine-tuned jointly
- Training loss: L_CTC + 0.1 * L_severity (ordinal EMD loss)

**Inputs:**
- `/kaggle/input/experiment-splits/` — torgo_train, torgo_test, ua_val, ua_test
- `/kaggle/input/b2-wavlm-ctc/b2v2_wavlm_ctc_final` — B2 model + processor

**Validation:** UASpeech val (800 stratified utterances) — same as B2, prevents TORGO prompt overfitting

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HOME"]            = "/kaggle/temp/hf"
os.environ["HF_DATASETS_CACHE"]  = "/kaggle/temp/hf/datasets"
os.environ["HF_HUB_CACHE"]       = "/kaggle/temp/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/temp/hf/transformers"
os.environ["XDG_CACHE_HOME"]     = "/kaggle/temp/.cache"

for p in [
    os.environ["HF_HOME"], os.environ["HF_DATASETS_CACHE"],
    os.environ["HF_HUB_CACHE"], os.environ["TRANSFORMERS_CACHE"],
    os.environ["XDG_CACHE_HOME"],
]:
    os.makedirs(p, exist_ok=True)

!rm -rf /kaggle/working/*
print("Cache dirs ready.")

Cache dirs ready.


In [2]:
!pip -q install -U transformers datasets accelerate evaluate jiwer soundfile packaging


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 108.6 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import evaluate
import json
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union
from itertools import groupby
from collections import Counter, defaultdict

from datasets import load_from_disk, concatenate_datasets
from transformers import (
    WavLMForCTC,
    WavLMModel,
    Wav2Vec2Processor,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
import transformers

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


transformers: 5.5.4
torch: 2.10.0+cu128
GPU: True
GPU name: Tesla T4


## Step 1 — Load splits and B2 processor

In [4]:
# ════════════════════════════════════════════════════════
# Split creation — inline, no external script needed
# Identical logic to create_splits.py
# ════════════════════════════════════════════════════════
import random
import pandas as pd
from collections import Counter
from datasets import load_dataset, load_from_disk

RANDOM_SEED          = 42
CONTROL_TRAIN_TARGET = 1500
UA_VAL_SAMPLE        = 800
TEST_SPEAKERS        = {"F01", "F04", "FC01", "M05"}

SEVERITY_MAPPING = {
    "F01": "severe",  "M01": "severe",  "M02": "severe",  "M04": "severe",
    "M05": "moderate", "F03": "moderate",
    "F04": "mild",    "M03": "mild",
    "FC01": "control", "FC02": "control", "FC03": "control",
    "MC01": "control", "MC02": "control", "MC03": "control", "MC04": "control",
}

# ── TORGO ──
print("Loading TORGO...")
cache = os.environ["HF_DATASETS_CACHE"]
ds   = load_dataset("abnerh/TORGO-database", cache_dir=cache)
df   = ds["train"].to_pandas()

df["audio_path"] = df["audio"].apply(lambda x: x["path"])
df["speaker"]    = df["audio_path"].apply(lambda p: str(p).split("_")[0])
df["Category"]   = df["speaker"].map(SEVERITY_MAPPING)

df["split"] = "train"
df.loc[df["speaker"].isin(TEST_SPEAKERS), "split"] = "test"

train_df      = df[df["split"] == "train"].copy()
train_control = train_df[train_df["Category"] == "control"]
train_dysarth = train_df[train_df["Category"] != "control"]

train_control_sampled = train_control.sample(
    n=min(CONTROL_TRAIN_TARGET, len(train_control)), random_state=RANDOM_SEED
)
train_df_final = pd.concat([train_dysarth, train_control_sampled]).sample(
    frac=1, random_state=RANDOM_SEED
)
test_df = df[df["split"] == "test"].copy()

hf_full     = ds["train"].add_column("speaker",  df["speaker"].tolist())
hf_full     = hf_full.add_column("Category", df["Category"].tolist())
torgo_train = hf_full.select(train_df_final.index.tolist())
torgo_test  = hf_full.select(test_df.index.tolist())

print(f"TORGO train: {len(torgo_train)}  test: {len(torgo_test)}")
print("Train severity:", dict(sorted(Counter(torgo_train["Category"]).items())))
print("Test severity: ", dict(sorted(Counter(torgo_test["Category"]).items())))
print("Train speakers:", sorted(set(torgo_train["speaker"])))
print("Test speakers: ", sorted(set(torgo_test["speaker"])))

# ── UASpeech ──
print("\nLoading UASpeech...")
ua_ds         = load_dataset("extraordinarylab/ua-speech", cache_dir=cache)
ua_train_full = ua_ds["train"]
ua_test_full  = ua_ds["test"]

random.seed(RANDOM_SEED)
severity_indices = {}
for i, sev in enumerate(ua_train_full["severity"]):
    severity_indices.setdefault(sev, []).append(i)

per_sev = UA_VAL_SAMPLE // len(severity_indices)
sampled = []
print(f"Stratified UASpeech val ({UA_VAL_SAMPLE} total):")
for sev, idxs in sorted(severity_indices.items()):
    n = min(per_sev, len(idxs))
    sampled.extend(random.sample(idxs, n))
    print(f"  {sev}: {n} from {len(idxs)}")

ua_val = ua_train_full.select(sorted(sampled))
print(f"UASpeech val: {len(ua_val)}  test: {len(ua_test_full)}")

# ── B2 path ──
B2_PATH = "/kaggle/input/datasets/kavishk/b2-pretrain/kaggle/working/b2v2_wavlm_ctc_final"
# Adjust B2_PATH to match your saved B2 model dataset path

processor = Wav2Vec2Processor.from_pretrained(B2_PATH)
print(f"\nProcessor loaded from: {B2_PATH}")
print(f"Vocab size: {processor.tokenizer.vocab_size}")

Loading TORGO...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/370M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/397M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/356M [00:00<?, ?B/s]

...

data/test-00002-of-00004.parquet:   0%|          | 0.00/512M [00:00<?, ?B/s]

data/test-00003-of-00004.parquet:   0%|          | 0.00/520M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/38656 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/18740 [00:00<?, ? examples/s]

Stratified UASpeech val (800 total):
  high: 200 from 6630
  low: 200 from 10606
  medium: 200 from 10710
  very low: 200 from 10710
UASpeech val: 800  test: 18740

Processor loaded from: /kaggle/input/datasets/kavishk/b2-pretrain/kaggle/working/b2v2_wavlm_ctc_final
Vocab size: 30


## Step 2 — Severity label mapping

TORGO uses: control, mild, moderate, severe (from `Category` column).
UASpeech uses its own severity labels — we map to the same 4-class scheme.
Integer encoding for ordinal loss: control=0, mild=1, moderate=2, severe=3.

In [5]:
# TORGO severity → integer
TORGO_SEV_TO_INT = {
    "control":  0,
    "mild":     1,
    "moderate": 2,
    "severe":   3,
}
INT_TO_SEV = {v: k for k, v in TORGO_SEV_TO_INT.items()}
NUM_SEV_CLASSES = 4

# UASpeech severity → integer
# extraordinarylab/ua-speech uses: 'high', 'medium', 'low', 'very low'
# Map to same 4-class ordinal scale
UA_SEV_TO_INT = {
    "high":     3,   # closest to control / mild
    "medium":   2,      # UASpeech uses "medium" not "mid"
    "low":      1,
    "very low": 0,   # most severe
}

# Sanity check: print unique UASpeech severities
ua_sevs = set(ua_val["severity"])
print("UASpeech severity values:", ua_sevs)
missing = ua_sevs - set(UA_SEV_TO_INT.keys())
if missing:
    print(f"WARNING: unmapped UASpeech severities: {missing}")
    print("Update UA_SEV_TO_INT above to cover these values.")
else:
    print("All UASpeech severity values mapped correctly.")

UASpeech severity values: {'very low', 'medium', 'high', 'low'}
All UASpeech severity values mapped correctly.


## Step 3 — Per-severity training subsets

Each LoRA adapter trains only on its own severity class.
We pre-split torgo_train into 4 subsets here for clarity.

In [6]:
# Split torgo_train by severity class
sev_subsets = {}
for sev in ["control", "mild", "moderate", "severe"]:
    subset = torgo_train.filter(
        lambda x: x["Category"] == sev,
        num_proc=1,
    )
    sev_subsets[sev] = subset
    print(f"  {sev:10s}: {len(subset)} utterances  "
          f"speakers: {sorted(set(subset['speaker']))}")

Filter (num_proc=1):   0%|          | 0/5578 [00:00<?, ? examples/s]

  control   : 1500 utterances  speakers: ['FC02', 'FC03', 'MC01', 'MC02', 'MC03', 'MC04']


Filter (num_proc=1):   0%|          | 0/5578 [00:00<?, ? examples/s]

  mild      : 810 utterances  speakers: ['M03']


Filter (num_proc=1):   0%|          | 0/5578 [00:00<?, ? examples/s]

  moderate  : 1087 utterances  speakers: ['F03']


Filter (num_proc=1):   0%|          | 0/5578 [00:00<?, ? examples/s]

  severe    : 2181 utterances  speakers: ['M01', 'M02', 'M04']


## Step 4 — Preprocessing functions

In [7]:
MAX_AUDIO_SAMPLES = 240_000  # 15 seconds at 16kHz

def prepare_torgo(batch):
    """Preprocess a TORGO example. Returns input_values, labels, severity_label."""
    audio = batch["audio"]
    arr   = audio["array"]

    # Truncate if needed
    if len(arr) > MAX_AUDIO_SAMPLES:
        arr = arr[:MAX_AUDIO_SAMPLES]

    inputs = processor(
        arr,
        sampling_rate=audio["sampling_rate"],
    )
    batch["input_values"]   = inputs.input_values[0]
    batch["labels"]         = processor.tokenizer(
        batch["transcription"].lower().strip()
    ).input_ids
    batch["severity_label"] = TORGO_SEV_TO_INT[batch["Category"]]
    return batch


def prepare_uaspeech(batch):
    """Preprocess a UASpeech example for validation."""
    audio = batch["audio"]
    arr   = audio["array"]

    if len(arr) > MAX_AUDIO_SAMPLES:
        arr = arr[:MAX_AUDIO_SAMPLES]

    inputs = processor(
        arr,
        sampling_rate=audio["sampling_rate"],
    )
    batch["input_values"]   = inputs.input_values[0]
    batch["labels"]         = processor.tokenizer(
        batch["text"].lower().strip()
    ).input_ids
    batch["severity_label"] = UA_SEV_TO_INT.get(batch["severity"], 3)
    return batch


print("Preprocessing functions defined.")

Preprocessing functions defined.


In [8]:
# Preprocess full torgo_train (all severities together — used for joint training)
TORGO_REMOVE = torgo_train.column_names  # remove all original columns

train_p = torgo_train.filter(
    lambda x: len(x["audio"]["array"]) <= MAX_AUDIO_SAMPLES,
    num_proc=1,
).map(
    prepare_torgo,
    remove_columns=TORGO_REMOVE,
    num_proc=1,
    desc="Preprocessing TORGO train",
)

UA_REMOVE = ua_val.column_names
val_p = ua_val.filter(
    lambda x: len(x["audio"]["array"]) <= MAX_AUDIO_SAMPLES,
    num_proc=1,
).map(
    prepare_uaspeech,
    remove_columns=UA_REMOVE,
    num_proc=1,
    desc="Preprocessing UASpeech val",
)

print(f"Train preprocessed: {len(train_p)}")
print(f"Val preprocessed:   {len(val_p)}")

# Sanity check
s = train_p[0]
print(f"\nSample input length:    {len(s['input_values'])}")
print(f"Sample labels decoded:  {processor.tokenizer.decode(s['labels'])}")
print(f"Sample severity_label:  {s['severity_label']} ({INT_TO_SEV[s['severity_label']]})")

Filter (num_proc=1):   0%|          | 0/5578 [00:00<?, ? examples/s]

Preprocessing TORGO train (num_proc=1):   0%|          | 0/5545 [00:00<?, ? examples/s]

Filter (num_proc=1):   0%|          | 0/800 [00:00<?, ? examples/s]

Preprocessing UASpeech val (num_proc=1):   0%|          | 0/797 [00:00<?, ? examples/s]

Train preprocessed: 5545
Val preprocessed:   797

Sample input length:    35680
Sample labels decoded:  slep
Sample severity_label:  1 (mild)


## Step 5 — Model architecture

### Design
- Load B2 WavLM backbone — frozen entirely
- Inject 4 LoRA adapter sets via PEFT (one per severity class)
- Severity classifier: MLP on mean-pooled layer-6 hidden states
- CTC head: initialised from B2 weights, trainable
- At training: severity label known from dataset → select correct adapter
- At inference: severity label from TORGO metadata → select correct adapter

### LoRA implementation strategy
PEFT's `get_peft_model` creates one set of LoRA matrices. For 4 separate adapters
we use PEFT's multi-adapter support: create 4 named adapters and switch between
them per batch based on the severity label of the samples in that batch.

Since each batch is single-severity (we group by severity during training),
only one adapter is active at a time — this is exact hard routing.

In [9]:
# ═══════════════════════════════════════════════════════════════
# Manual LoRA implementation — no PEFT dependency
# ═══════════════════════════════════════════════════════════════

class LoRALinear(nn.Module):
    """
    Drop-in replacement for nn.Linear with a LoRA side-path.

    Forward:
        output = x @ W.T + x @ (A @ B).T * (alpha / rank)

    W is frozen. A and B are trainable.
    Standard LoRA init: A ~ N(0, 1), B = 0
    so the adapter output is zero at the start of training.
    """
    def __init__(self, linear: nn.Linear, rank: int, alpha: int):
        super().__init__()
        in_features  = linear.in_features
        out_features = linear.out_features

        # Frozen original weight
        self.weight = linear.weight  # [out, in]
        self.bias   = linear.bias    # [out] or None
        # Freeze them explicitly
        self.weight.requires_grad = False
        if self.bias is not None:
            self.bias.requires_grad = False

        # Trainable LoRA matrices
        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        self.scale  = alpha / rank

        # Standard LoRA init: A from N(0,1), B=0
        nn.init.normal_(self.lora_A)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base = F.linear(x, self.weight, self.bias)
        lora = F.linear(F.linear(x, self.lora_A), self.lora_B) * self.scale
        return base + lora

    def extra_repr(self):
        return (f"in={self.weight.shape[1]}, out={self.weight.shape[0]}, "
                f"rank={self.lora_A.shape[0]}, scale={self.scale:.3f}")


def inject_lora(wavlm_encoder: nn.Module, rank: int, alpha: int) -> nn.Module:
    """
    Replace q_proj and v_proj in all WavLM attention layers with LoRALinear.
    Returns the modified module in-place.
    """
    replaced = 0
    for name, module in wavlm_encoder.named_modules():
        # WavLM attention: encoder.layers.N.attention.q_proj / v_proj
        if not isinstance(module, nn.MultiheadAttention):
            continue
        parent_name = name  # e.g. "encoder.layers.0.attention"
        # Navigate to the parent module
        parts  = name.split(".")
        parent = wavlm_encoder
        for p in parts:
            parent = getattr(parent, p)

        for proj in ["q_proj", "v_proj"]:
            if hasattr(parent, proj):
                original = getattr(parent, proj)
                if isinstance(original, nn.Linear):
                    setattr(parent, proj, LoRALinear(original, rank, alpha))
                    replaced += 1

    print(f"  Replaced {replaced} linear layers with LoRALinear (rank={rank}, alpha={alpha})")
    return wavlm_encoder


class WavLMWithLoRA(nn.Module):
    """
    WavLM encoder with one set of LoRA matrices active at a time.
    Wraps a standard WavLMModel and injects LoRALinear into all attention layers.
    Maintains N_ADAPTERS separate (lora_A, lora_B) parameter sets — one per severity.
    At any time, exactly one adapter's parameters are loaded into the LoRALinear layers.
    """

    SEV_NAMES = ["control", "mild", "moderate", "severe"]
    N_ADAPTERS = 4

    def __init__(self, wavlm_model: nn.Module, rank: int, alpha: int):
        super().__init__()

        # Freeze backbone transformer (CNN already frozen externally)
        # We will selectively unfreeze via optimizer groups
        self.encoder = wavlm_model

        # Inject LoRA into q_proj and v_proj
        self.encoder = inject_lora(self.encoder, rank, alpha)

        # Collect all LoRALinear layers for adapter management
        self._lora_layers = [
            m for m in self.encoder.modules()
            if isinstance(m, LoRALinear)
        ]
        n_lora = len(self._lora_layers)
        print(f"  Found {n_lora} LoRALinear layers")

        # Allocate N_ADAPTERS separate parameter sets
        # Each adapter: list of (A, B) pairs, one per LoRALinear layer
        self.adapter_A = nn.ParameterList([
            nn.ParameterList([
                nn.Parameter(layer.lora_A.data.clone())
                for layer in self._lora_layers
            ])
            for _ in range(self.N_ADAPTERS)
        ])
        self.adapter_B = nn.ParameterList([
            nn.ParameterList([
                nn.Parameter(layer.lora_B.data.clone())
                for layer in self._lora_layers
            ])
            for _ in range(self.N_ADAPTERS)
        ])

        # Load adapter 0 (control) into LoRALinear layers at init
        self._active_adapter = 0
        self._load_adapter(0)

    def _load_adapter(self, adapter_idx: int):
        """Copy adapter_idx's (A, B) params into the live LoRALinear layers."""        
        for i, layer in enumerate(self._lora_layers):
            layer.lora_A.data.copy_(self.adapter_A[adapter_idx][i].data)
            layer.lora_B.data.copy_(self.adapter_B[adapter_idx][i].data)
        self._active_adapter = adapter_idx

    def _save_adapter(self, adapter_idx: int):
        """Save current LoRALinear params back into the adapter store."""        
        for i, layer in enumerate(self._lora_layers):
            self.adapter_A[adapter_idx][i].data.copy_(layer.lora_A.data)
            self.adapter_B[adapter_idx][i].data.copy_(layer.lora_B.data)

    def set_active_adapter(self, adapter_idx: int):
        """Switch to adapter_idx. Saves current adapter first."""        
        if adapter_idx == self._active_adapter:
            return
        self._save_adapter(self._active_adapter)
        self._load_adapter(adapter_idx)

    def get_all_adapter_outputs(
        self,
        input_values: torch.Tensor,
        attention_mask: Optional[torch.Tensor],
    ) -> list:
        """Run all 4 adapters, return list of last_hidden_state tensors."""        
        outputs = []
        current = self._active_adapter
        for idx in range(self.N_ADAPTERS):
            self.set_active_adapter(idx)
            out = self.encoder(
                input_values,
                attention_mask=attention_mask,
                output_hidden_states=False,
            )
            outputs.append(out.last_hidden_state.detach())
        self.set_active_adapter(current)
        return outputs

    def forward(self, input_values, attention_mask=None, output_hidden_states=False):
        return self.encoder(
            input_values,
            attention_mask=attention_mask,
            output_hidden_states=output_hidden_states,
        )


class WavLMLoRASeverity(nn.Module):
    """
    WavLM-base with severity-stratified LoRA adapters (manual implementation).

    Changes from v3:
      1. Backbone unfrozen with low LR (5e-6) — SAT from Hu et al. 2025
      2. Diversity loss between adapter pairs (cosine similarity, weight=0.1)
      3. LoRA rank=16 (was 8)
      4. No PEFT dependency — LoRA implemented manually
    """

    SEV_NAMES = ["control", "mild", "moderate", "severe"]

    def __init__(self, b2_model_path: str, lora_rank: int = 16, lora_alpha: int = 32):
        super().__init__()

        # Load B2 WavLMForCTC
        base        = WavLMForCTC.from_pretrained(
            b2_model_path,
            ctc_loss_reduction="mean",
            ctc_zero_infinity=True,
        )
        vocab_size  = base.config.vocab_size
        hidden_size = base.config.hidden_size  # 768

        # Save B2 CTC head weights
        ctc_weight = base.lm_head.weight.data.clone()
        ctc_bias   = base.lm_head.bias.data.clone() if base.lm_head.bias is not None else None

        # Freeze CNN feature extractor
        for param in base.wavlm.feature_extractor.parameters():
            param.requires_grad = False

        # Unfreeze transformer layers (will train at low LR via optimizer groups)
        for param in base.wavlm.encoder.parameters():
            param.requires_grad = True

        # Wrap WavLM encoder with manual LoRA
        print("Injecting LoRA adapters...")
        self.wavlm_lora = WavLMWithLoRA(base.wavlm, lora_rank, lora_alpha)

        # CTC head — initialised from B2 weights, trainable
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=(ctc_bias is not None))
        self.lm_head.weight.data.copy_(ctc_weight)
        if ctc_bias is not None:
            self.lm_head.bias.data.copy_(ctc_bias)

        # Severity classifier — random init, trainable
        self.severity_classifier = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, NUM_SEV_CLASSES),
        )

        self.hidden_size   = hidden_size
        self.vocab_size    = vocab_size
        self.pad_token_id  = base.config.pad_token_id
        self.config        = base.config
        self.lora_rank     = lora_rank
        self._active_sev   = 0

        self._print_param_summary()

    def _print_param_summary(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)

        # LoRA params: adapter_A and adapter_B across all adapters
        lora_params = sum(
            p.numel() for n, p in self.named_parameters()
            if "adapter_A" in n or "adapter_B" in n
        )
        # Backbone params: encoder weights (frozen LoRALinear base weights + unfrozen transformer)
        backbone_params = sum(
            p.numel() for n, p in self.wavlm_lora.encoder.named_parameters()
            if "lora_A" not in n and "lora_B" not in n and p.requires_grad
        )
        print(f"Total parameters:        {total:,}")
        print(f"Trainable parameters:    {trainable:,}")
        print(f"  Backbone transformer:  {backbone_params:,}  (lr=5e-6)")
        print(f"  LoRA (all adapters):   {lora_params:,}  (lr=1e-4)")
        print(f"  CTC head:              {sum(p.numel() for p in self.lm_head.parameters()):,}  (lr=1e-4)")
        print(f"  Severity classifier:   {sum(p.numel() for p in self.severity_classifier.parameters()):,}  (lr=1e-4)")

    def set_active_severity(self, sev_int: int):
        """Switch the active LoRA adapter."""        
        
        self.wavlm_lora.set_active_adapter(sev_int)
        self._active_sev = sev_int

    def forward(
        self,
        input_values: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        severity_label: Optional[torch.Tensor] = None,
        compute_kl: bool = False,
    ):
        # WavLM forward
        wavlm_out  = self.wavlm_lora(
            input_values,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        hidden     = wavlm_out.last_hidden_state   # [B, T, 768]
        all_hidden = wavlm_out.hidden_states        # CNN + 12 transformer layers

        # CTC logits
        logits = self.lm_head(hidden)  # [B, T, vocab]

        # Severity classifier from layer 6
        layer6 = all_hidden[7]
        if attention_mask is not None:
            feat_lengths = self.wavlm_lora.encoder._get_feat_extract_output_lengths(
                attention_mask.sum(-1)
            ).long()
            T    = layer6.size(1)
            mask = (torch.arange(T, device=layer6.device).unsqueeze(0)
                    < feat_lengths.unsqueeze(1)).unsqueeze(-1).float()
            pooled = (layer6 * mask).sum(1) / mask.sum(1).clamp(min=1)
        else:
            pooled = layer6.mean(dim=1)
        sev_logits = self.severity_classifier(pooled)

        # Losses
        loss = ctc_loss_val = sev_loss_val = kl_loss_val = None

        if labels is not None:
            log_probs     = F.log_softmax(logits, dim=-1).transpose(0, 1)
            input_lengths = torch.full(
                (logits.size(0),), logits.size(1),
                dtype=torch.long, device=logits.device,
            )
            label_lengths = (labels != -100).sum(-1)
            labels_ctc    = labels.clone()
            labels_ctc[labels_ctc == -100] = self.pad_token_id

            ctc_loss_val = F.ctc_loss(
                log_probs, labels_ctc, input_lengths, label_lengths,
                blank=self.pad_token_id, reduction="mean", zero_infinity=True,
            )
            loss = ctc_loss_val

            if severity_label is not None:
                sev_loss_val = emd_loss(sev_logits, severity_label)
                loss = loss + 0.1 * sev_loss_val

            if compute_kl:
                adapter_outputs = self.wavlm_lora.get_all_adapter_outputs(
                    input_values, attention_mask
                )
                kl_loss_val = diversity_loss(adapter_outputs)
                # Weight 0.1 — same scale as severity loss, keeps CTC dominant
                loss = loss + 0.1 * kl_loss_val

        return LoRASeverityOutput(
            loss=loss,
            logits=logits,
            ctc_loss=ctc_loss_val,
            sev_loss=sev_loss_val,
            kl_loss=kl_loss_val,
            sev_logits=sev_logits,
        )


class LoRASeverityOutput:
    def __init__(self, loss, logits, ctc_loss, sev_loss, kl_loss, sev_logits):
        self.loss       = loss
        self.logits     = logits
        self.ctc_loss   = ctc_loss
        self.sev_loss   = sev_loss
        self.kl_loss    = kl_loss
        self.sev_logits = sev_logits
        self._data      = (loss, logits)

    def __getitem__(self, key):
        return self._data[key]

    def __iter__(self):
        yield self.loss
        yield self.logits

    def __len__(self):
        return len(self._data)


print("Model classes defined (manual LoRA, no PEFT).")


Model classes defined (manual LoRA, no PEFT).


## Step 6 — Ordinal EMD loss

In [10]:
def emd_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    Earth Mover's Distance loss for ordinal severity classification.
    Penalises misclassifications proportionally to distance on:
      control(0) < mild(1) < moderate(2) < severe(3)
    """
    probs      = F.softmax(logits, dim=-1)
    target_one = F.one_hot(targets, num_classes=NUM_SEV_CLASSES).float()
    pred_cdf   = torch.cumsum(probs,      dim=-1)
    target_cdf = torch.cumsum(target_one, dim=-1)
    return torch.abs(pred_cdf - target_cdf).sum(dim=-1).mean()


def diversity_loss(adapter_outputs: list) -> torch.Tensor:
    """
    Diversity regularisation loss inspired by Hu et al. 2025.

    Encourages adapter outputs to be different from each other by
    minimising the average pairwise cosine similarity between
    mean-pooled adapter representations.

    Range: [-1, 1] in theory, [0, 1] in practice for hidden states.
    A value near 0 means adapters are already orthogonal (diverse).
    A value near 1 means adapters are producing identical representations.

    Weighted at 0.1 in total loss — same scale as severity EMD loss.
    This keeps it as a soft regulariser without overriding CTC.

    adapter_outputs: list of 4 tensors [B, T, 768], one per adapter.
    """
    # Mean-pool over time dimension: [B, 768] per adapter
    pooled = [o.mean(dim=1) for o in adapter_outputs]

    # L2-normalise so cosine similarity = dot product
    normed = [F.normalize(p, dim=-1) for p in pooled]

    loss  = torch.tensor(0.0, device=adapter_outputs[0].device)
    count = 0
    n     = len(normed)

    for i in range(n):
        for j in range(i + 1, n):  # upper triangle — each pair counted once
            cos_sim = (normed[i] * normed[j]).sum(dim=-1).mean()  # scalar in [-1, 1]
            loss    = loss + cos_sim
            count  += 1

    # Average over all pairs: 4 adapters → 6 pairs
    # Minimising this pushes adapters to be orthogonal in representation space
    return loss / count


# ── Sanity checks ──
with torch.no_grad():
    l_correct  = emd_loss(torch.tensor([[0., 0., 0., 10.]]), torch.tensor([3]))
    l_adjacent = emd_loss(torch.tensor([[0., 0., 10., 0.]]), torch.tensor([3]))
    l_far      = emd_loss(torch.tensor([[10., 0., 0., 0.]]), torch.tensor([3]))
    print("EMD sanity check:")
    print(f"  correct:  {l_correct.item():.4f}  adjacent: {l_adjacent.item():.4f}  far: {l_far.item():.4f}")
    assert l_correct < l_adjacent < l_far
    print("  Ordering correct.")

    # Diversity loss: identical adapters should give ~1.0, random should give ~0
    identical = [torch.ones(2, 10, 768)] * 4
    div_same  = diversity_loss(identical)
    diverse   = [torch.randn(2, 10, 768) for _ in range(4)]
    div_rand  = diversity_loss(diverse)
    print(f"Diversity loss — identical adapters: {div_same.item():.4f}  (should be ~1.0)")
    print(f"Diversity loss — random adapters:    {div_rand.item():.4f}  (should be near 0)")
    assert div_same > div_rand, "Diversity loss ordering check failed"
    print("  Ordering correct.")

print("Losses defined.")


EMD sanity check:
  correct:  0.0003  adjacent: 1.0001  far: 2.9997
  Ordering correct.
Diversity loss — identical adapters: 1.0000  (should be ~1.0)
Diversity loss — random adapters:    0.0036  (should be near 0)
  Ordering correct.
Losses defined.


## Step 7 — Data Collator

Same as B2 but also handles `severity_label` column.

In [11]:
@dataclass
class DataCollatorLoRA:
    """
    Pads input_values and labels for CTC.
    Also collects severity_label as a tensor.
    """
    processor: Any
    padding: Union[bool, str] = "longest"

    def __call__(
        self,
        features: List[Dict[str, Any]],
    ) -> Dict[str, torch.Tensor]:

        # Pad input_values
        input_feats = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_feats,
            padding=self.padding,
            return_tensors="pt",
        )

        # Pad labels with -100
        label_feats = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_feats,
            padding=self.padding,
            return_tensors="pt",
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels

        # Severity labels as long tensor
        if "severity_label" in features[0]:
            batch["severity_label"] = torch.tensor(
                [f["severity_label"] for f in features],
                dtype=torch.long,
            )

        return batch


data_collator = DataCollatorLoRA(processor=processor)
print("Data collator ready.")

Data collator ready.


## Step 8 — Custom Trainer with severity-aware batch routing

The key challenge: each batch must contain only one severity class
so the correct adapter is active for the entire batch.

We solve this with a custom Trainer that:
1. Groups training samples by severity class
2. Before each forward pass, calls set_active_severity() based on
   the severity_label of the batch
3. Also logs ctc_loss and sev_loss separately for monitoring

In [12]:
from torch.utils.data import Sampler, DataLoader
import math


class SeverityGroupedSampler(Sampler):
    """
    Yields batches where all samples share the same severity class.
    Shuffles within each severity group and shuffles the order of groups.
    This ensures set_active_severity() is called correctly per batch.
    """

    def __init__(
        self,
        dataset,
        batch_size: int,
        seed: int = 42,
        drop_last: bool = False,
    ):
        self.batch_size = batch_size
        self.seed       = seed
        self.drop_last  = drop_last

        # Group indices by severity_label
        self.groups = defaultdict(list)
        for idx in range(len(dataset)):
            sev = dataset[idx]["severity_label"]
            self.groups[sev].append(idx)

        print("SeverityGroupedSampler group sizes:")
        for sev, idxs in sorted(self.groups.items()):
            print(f"  {INT_TO_SEV[sev]:10s}: {len(idxs)} samples")

    def __iter__(self):
        import random as rng
        rng.seed(self.seed)

        # Shuffle within each group
        all_batches = []
        for sev, idxs in self.groups.items():
            shuffled = idxs.copy()
            rng.shuffle(shuffled)
            # Chunk into batches of batch_size
            for start in range(0, len(shuffled), self.batch_size):
                chunk = shuffled[start : start + self.batch_size]
                if self.drop_last and len(chunk) < self.batch_size:
                    continue
                all_batches.append(chunk)

        # Shuffle the order of batches across severity groups
        rng.shuffle(all_batches)

        for batch in all_batches:
            yield from batch

    def __len__(self):
        total = 0
        for idxs in self.groups.values():
            n_batches = len(idxs) // self.batch_size
            if not self.drop_last and len(idxs) % self.batch_size > 0:
                n_batches += 1
            total += n_batches * self.batch_size
        return total


class LoRATrainer(Trainer):
    """
    Custom Trainer with:
    1. SeverityGroupedSampler — single-severity batches
    2. set_active_severity() before each forward pass
    3. Differential LRs: backbone=5e-6, LoRA/head/classifier=1e-4
    4. KL diversity loss during training (compute_kl=True)
    """

    def get_train_dataloader(self) -> DataLoader:
        sampler = SeverityGroupedSampler(
            self.train_dataset,
            batch_size=self.args.per_device_train_batch_size,
            seed=self.args.seed,
            drop_last=self.args.dataloader_drop_last,
        )
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.per_device_train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
        )

    def create_optimizer(self):
        model        = self.model
        backbone_lr  = 5e-6
        adapter_lr   = 1e-4

        backbone_params   = []
        lora_params       = []
        head_params       = list(model.lm_head.parameters())
        cls_params        = list(model.severity_classifier.parameters())

        for name, param in model.wavlm_lora.named_parameters():
            if not param.requires_grad:
                continue
            if "adapter_A" in name or "adapter_B" in name:
                lora_params.append(param)
            elif "lora_A" in name or "lora_B" in name:
                # Live LoRALinear weights — these are aliases of adapter params
                # Skip to avoid double-counting (adapter_A/B are the real params)
                pass
            else:
                backbone_params.append(param)

        param_groups = [
            {"params": backbone_params, "lr": backbone_lr, "name": "backbone"},
            {"params": lora_params,     "lr": adapter_lr,  "name": "lora_adapters"},
            {"params": head_params,     "lr": adapter_lr,  "name": "ctc_head"},
            {"params": cls_params,      "lr": adapter_lr,  "name": "severity_cls"},
        ]
        param_groups = [g for g in param_groups if len(g["params"]) > 0]

        print("Optimizer parameter groups:")
        for g in param_groups:
            n = sum(p.numel() for p in g["params"])
            print(f"  {g['name']:20s}: {n:>10,} params  lr={g['lr']}")

        self.optimizer = torch.optim.AdamW(
            param_groups,
            weight_decay=self.args.weight_decay,
        )
        return self.optimizer

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        severity_label = inputs.pop("severity_label", None)

        if severity_label is not None:
            model.set_active_severity(severity_label[0].item())
        inputs["severity_label"] = severity_label
        inputs["compute_kl"]     = True

        outputs = model(**inputs)
        loss    = outputs.loss

        log_dict = {}
        if outputs.ctc_loss is not None:
            log_dict["ctc_loss"] = outputs.ctc_loss.item()
        if outputs.sev_loss is not None:
            log_dict["sev_loss"]     = outputs.sev_loss.item()
            log_dict["sev_weighted"] = (0.1 * outputs.sev_loss).item()
        if outputs.kl_loss is not None:
            log_dict["kl_loss"]     = outputs.kl_loss.item()
            log_dict["kl_weighted"] = (0.1 * outputs.kl_loss).item()
        if log_dict:
            self.log(log_dict)

        return (loss, outputs) if return_outputs else loss


print("Custom trainer and sampler defined.")


Custom trainer and sampler defined.


## Step 9 — Metrics

In [13]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def decode_ctc(pred_ids, tokenizer):
    blank_id = tokenizer.pad_token_id
    decoded = []
    for seq in pred_ids:
        collapsed = [k for k, _ in groupby(seq)]
        filtered = [t for t in collapsed if t != blank_id]
        decoded.append(
            tokenizer.decode(filtered, skip_special_tokens=True)
            if filtered else ""
        )
    return decoded


def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    label_ids = pred.label_ids
    pad_id = processor.tokenizer.pad_token_id

    # Handle jagged/tuple label structure from custom trainer
    if isinstance(label_ids, tuple):
        label_ids = label_ids[0]
    if isinstance(label_ids, list):
        label_ids = np.array(label_ids)
    if label_ids.dtype == object:
        # Jagged array — pad manually
        max_len = max(len(seq) for seq in label_ids)
        padded = np.full((len(label_ids), max_len), -100, dtype=np.int64)
        for i, seq in enumerate(label_ids):
            seq = np.array(seq)
            padded[i, :len(seq)] = seq
        label_ids = padded

    # Replace -100 with pad token
    label_ids = np.where(label_ids != -100, label_ids, pad_id)

    pred_str = decode_ctc(pred_ids, processor.tokenizer)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [p.strip() if p.strip() else "⟨empty⟩" for p in pred_str]
    label_str = [l.strip() for l in label_str]

    # Guard against length mismatch
    min_len = min(len(pred_str), len(label_str))
    pred_str = pred_str[:min_len]
    label_str = label_str[:min_len]

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": round(wer, 4), "cer": round(cer, 4)}

print("Metrics ready.")

Metrics ready.


## Step 10 — Instantiate model

In [14]:
model = WavLMLoRASeverity(
    b2_model_path=B2_PATH,
    lora_rank=16,    # increased from 8 — more expressive capacity per adapter
    lora_alpha=32,   # alpha = 2 * rank (standard convention)
)
# gradient_checkpointing disabled — PEFT + unfrozen backbone handles memory
model.gradient_checkpointing_enable = lambda **kw: None
print("Model instantiated.")


Loading weights:   0%|          | 0/250 [00:00<?, ?it/s]

Injecting LoRA adapters...
  Replaced 0 linear layers with LoRALinear (rank=16, alpha=32)
  Found 0 LoRALinear layers
Total parameters:        94,505,492
Trainable parameters:    90,305,044
  Backbone transformer:  90,181,488  (lr=5e-6)
  LoRA (all adapters):   0  (lr=1e-4)
  CTC head:              24,608  (lr=1e-4)
  Severity classifier:   98,948  (lr=1e-4)
Model instantiated.


## Step 11 — Training arguments

In [15]:
CHECKPOINT_DIR = "/kaggle/working/p1_lora_severity"

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    num_train_epochs=30,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,    # effective batch = 16

    # LR set to backbone LR here — LoRA/head LR managed via create_optimizer()
    # This value is used as the base for the LR scheduler
    learning_rate=5e-6,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,

    fp16=True,
    dataloader_num_workers=2,
    dataloader_pin_memory=False,

    logging_steps=25,
    logging_dir="/kaggle/temp/tb_logs",
    report_to="none",

    save_total_limit=2,
    seed=42,
)

print("Training arguments set.")
print(f"Effective batch size: "
      f"{training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments set.
Effective batch size: 16


In [16]:
class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        if "loss" in logs:
            print(f"[step {state.global_step}] total={logs['loss']:.4f}", end="")
            if "ctc_loss" in logs:
                print(f"  ctc={logs['ctc_loss']:.4f}", end="")
            if "sev_weighted" in logs:
                print(f"  sev(x0.1)={logs['sev_weighted']:.4f}", end="")
            if "kl_weighted" in logs:
                print(f"  div(x0.1)={logs['kl_weighted']:.4f}", end="")
            print()
        if "eval_cer" in logs:
            print(f"[step {state.global_step}] val_cer={logs['eval_cer']:.4f}  "
                  f"val_wer={logs.get('eval_wer', 'N/A')}")


class TrainLossEarlyStoppingCallback(TrainerCallback):
    """
    Early stopping on epoch-average training loss.
    Saves best weights in memory and restores them at end of training.
    """
    def __init__(self, patience: int = 5, threshold: float = 0.005):
        self.patience          = patience
        self.threshold         = threshold
        self.best_loss         = float("inf")
        self.epochs_no_improve = 0
        self.best_state        = None

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        # Compute epoch-average loss from all step-level log entries this epoch
        current_epoch = int(state.epoch)
        step_losses   = [
            e["loss"] for e in state.log_history
            if "loss" in e
            and "eval_loss" not in e
            and e.get("epoch", -1) >= current_epoch - 1
            and e.get("epoch", -1) < current_epoch
        ]
        if not step_losses:
            # Fallback: most recent step loss
            for e in reversed(state.log_history):
                if "loss" in e and "eval_loss" not in e:
                    step_losses = [e["loss"]]
                    break
        if not step_losses:
            return control

        train_loss = sum(step_losses) / len(step_losses)
        print(f"  [EarlyStopping] Epoch {current_epoch} avg loss: {train_loss:.6f} "
              f"(from {len(step_losses)} steps)", flush=True)

        if train_loss < self.best_loss - self.threshold:
            self.best_loss         = train_loss
            self.epochs_no_improve = 0
            self.best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            print(f"  [EarlyStopping] New best: {self.best_loss:.6f}. Weights saved.",
                  flush=True)
        else:
            self.epochs_no_improve += 1
            print(f"  [EarlyStopping] No improvement {self.epochs_no_improve}/{self.patience}. "
                  f"Best: {self.best_loss:.6f}", flush=True)
            if self.epochs_no_improve >= self.patience:
                print("  [EarlyStopping] Stopping.", flush=True)
                control.should_training_stop = True
        return control

    def on_train_end(self, args, state, control, model=None, **kwargs):
        if self.best_state is not None:
            device = next(model.parameters()).device
            model.load_state_dict({k: v.to(device) for k, v in self.best_state.items()})
            print(f"  [EarlyStopping] Best weights restored "
                  f"(avg train loss: {self.best_loss:.6f}).", flush=True)
        return control


train_early_stopping = TrainLossEarlyStoppingCallback(patience=5, threshold=0.005)
print("Callbacks ready.")


Callbacks ready.


## Step 12 — Train

In [17]:
trainer = LoRATrainer(
    model=model,
    args=training_args,
    train_dataset=train_p,
    eval_dataset=val_p,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[PrintLossCallback(), train_early_stopping],
)

print("\n--- DISK USAGE BEFORE TRAINING ---")
!du -sh /kaggle/working/* 2>/dev/null || true

print("\nStarting P1 v4 training...")
print("Changes from v3:")
print("  - Backbone unfrozen (backbone lr=5e-6, LoRA/head lr=1e-4)")
print("  - Diversity loss between adapters (cosine similarity, weight=0.1)")
print("  - LoRA rank 16 (was 8)")
print("  - Epoch-average loss for early stopping (was single-step)")
print("  - Best weights restored at end of training")
trainer.train()
print("\nTraining complete. Best model weights are now active.")



--- DISK USAGE BEFORE TRAINING ---
500K	/kaggle/working/__notebook__.ipynb
4.0K	/kaggle/working/p1_lora_severity

Starting P1 v4 training...
Changes from v3:
  - Backbone unfrozen (backbone lr=5e-6, LoRA/head lr=1e-4)
  - Diversity loss between adapters (cosine similarity, weight=0.1)
  - LoRA rank 16 (was 8)
  - Epoch-average loss for early stopping (was single-step)
  - Best weights restored at end of training
SeverityGroupedSampler group sizes:
  control   : 1500 samples
  mild      : 810 samples
  moderate  : 1086 samples
  severe    : 2149 samples
Optimizer parameter groups:
  backbone            : 90,181,488 params  lr=5e-06
  ctc_head            :     24,608 params  lr=0.0001
  severity_cls        :     98,948 params  lr=0.0001


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Epoch,Training Loss,Validation Loss,Wer,Cer
0,0.769904,4.056434,1.072800,0.726600
1,0.599093,4.293771,1.085300,0.729200
2,0.547516,4.479992,1.079000,0.734800
3,0.544331,4.536871,1.080300,0.734400
4,0.490896,4.683153,1.084100,0.741700
5,0.445506,4.624934,1.099100,0.739600
6,0.494295,4.872669,1.101600,0.750600
7,0.509547,4.824374,1.102900,0.742400
8,0.398520,5.041410,1.110400,0.749700
9,0.428323,4.954889,1.086600,0.745200


[step 25] total=0.7246
[step 50] total=0.7092
[step 75] total=0.7513
[step 100] total=0.7518
[step 125] total=0.7443
[step 150] total=0.7289
[step 175] total=0.7264
[step 200] total=0.7543
[step 225] total=0.7040
[step 250] total=0.7866
[step 275] total=0.7404
[step 300] total=0.7522
[step 325] total=0.7699
  [EarlyStopping] Epoch 0 avg loss: 0.769904 (from 1 steps)
  [EarlyStopping] New best: 0.769904. Weights saved.
[step 346] val_cer=0.7266  val_wer=1.0728


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


...

[step 8675] total=0.3604
[step 8700] total=0.3910
[step 8725] total=0.4107
[step 8750] total=0.3507
[step 8775] total=0.3621
[step 8800] total=0.3319
[step 8825] total=0.4340
[step 8850] total=0.4132
[step 8875] total=0.3731
[step 8900] total=0.3891
[step 8925] total=0.4047
[step 8950] total=0.3732
[step 8975] total=0.4194
  [EarlyStopping] Epoch 25 avg loss: 0.372665 (from 14 steps)
  [EarlyStopping] No improvement 3/5. Best: 0.370246
[step 8996] val_cer=0.7483  val_wer=1.0878


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


[step 9000] total=0.3655
[step 9025] total=0.3333
[step 9050] total=0.3617
[step 9075] total=0.3905
[step 9100] total=0.3371
[step 9125] total=0.3146
[step 9150] total=0.4248
[step 9175] total=0.4139
[step 9200] total=0.4011
[step 9225] total=0.3525
[step 9250] total=0.3484
[step 9275] total=0.3532
[step 9300] total=0.3762
[step 9325] total=0.4171
  [EarlyStopping] Epoch 26 avg loss: 0.385654 (from 13 steps)
  [EarlyStopping] No improvement 4/5. Best: 0.370246
[step 9342] val_cer=0.7511  val_wer=1.1004


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


[step 9350] total=0.3112
[step 9375] total=0.3443
[step 9400] total=0.3323
[step 9425] total=0.4505
[step 9450] total=0.3597
[step 9475] total=0.4673
[step 9500] total=0.3467
[step 9525] total=0.4017
[step 9550] total=0.4658
[step 9575] total=0.3523
[step 9600] total=0.3641
[step 9625] total=0.3963
[step 9650] total=0.3582
[step 9675] total=0.4397
  [EarlyStopping] Epoch 27 avg loss: 0.370714 (from 14 steps)
  [EarlyStopping] No improvement 5/5. Best: 0.370246
  [EarlyStopping] Stopping.
[step 9688] val_cer=0.7511  val_wer=1.0991
  [EarlyStopping] Best weights restored (avg train loss: 0.370246).

Training complete. Best model weights are now active.


In [18]:
FINAL_MODEL_PATH = "/kaggle/working/p1_lora_severity_final"
os.makedirs(FINAL_MODEL_PATH, exist_ok=True)

print(f"Saving best model (avg train loss: {train_early_stopping.best_loss:.6f}) "
      f"to {FINAL_MODEL_PATH}")

# Save full model state dict (backbone + LoRA + CTC head + classifier)
torch.save(model.state_dict(),
           os.path.join(FINAL_MODEL_PATH, "model_state.pt"))

# Save CTC head and classifier separately for convenience
torch.save(model.lm_head.state_dict(),
           os.path.join(FINAL_MODEL_PATH, "lm_head.pt"))
torch.save(model.severity_classifier.state_dict(),
           os.path.join(FINAL_MODEL_PATH, "severity_classifier.pt"))

# Save processor
processor.save_pretrained(FINAL_MODEL_PATH)

# Save config
with open(os.path.join(FINAL_MODEL_PATH, "severity_config.json"), "w") as f:
    json.dump({
        "sev_to_int":         TORGO_SEV_TO_INT,
        "int_to_sev":         {str(k): v for k, v in INT_TO_SEV.items()},
        "adapter_names":      WavLMLoRASeverity.SEV_NAMES,
        "lora_rank":          16,
        "lora_alpha":         32,
        "backbone_lr":        5e-6,
        "adapter_lr":         1e-4,
        "diversity_loss_weight": 0.1,
        "sev_loss_weight":    0.1,
        "best_train_loss":    round(train_early_stopping.best_loss, 6),
    }, f, indent=2)

print(f"Saved to: {FINAL_MODEL_PATH}")
!du -sh /kaggle/working/* 2>/dev/null || true


Saving best model (avg train loss: 0.370246) to /kaggle/working/p1_lora_severity_final
Saved to: /kaggle/working/p1_lora_severity_final
576K	/kaggle/working/__notebook__.ipynb
2.1G	/kaggle/working/p1_lora_severity
362M	/kaggle/working/p1_lora_severity_final


## Step 13 — Evaluate on TORGO test set

At test time: severity label is known from TORGO metadata.
We use the metadata label directly to route each speaker to the
correct adapter — no classifier needed.

In [19]:
# Preprocess test set
test_cats = torgo_test["Category"]

test_p = torgo_test.filter(
    lambda x: len(x["audio"]["array"]) <= MAX_AUDIO_SAMPLES,
    num_proc=1,
).map(
    prepare_torgo,
    remove_columns=torgo_test.column_names,
    num_proc=1,
    desc="Preprocessing TORGO test",
)

# Re-fetch categories after filter (order preserved)
test_cats_filtered = torgo_test.filter(
    lambda x: len(x["audio"]["array"]) <= MAX_AUDIO_SAMPLES,
    num_proc=1,
)["Category"]

print(f"TORGO test preprocessed: {len(test_p)}")

Filter (num_proc=1):   0%|          | 0/1798 [00:00<?, ? examples/s]

Preprocessing TORGO test (num_proc=1):   0%|          | 0/1786 [00:00<?, ? examples/s]

TORGO test preprocessed: 1786


In [20]:
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

all_preds  = []
all_labels = []
all_cats   = []

TEST_BATCH_SIZE = 8

for i in range(0, len(test_p), TEST_BATCH_SIZE):
    end = min(i + TEST_BATCH_SIZE, len(test_p))

    # All samples in this batch — group by severity for correct routing
    # At test time we process all severities together but route per-sample
    # Since each test speaker has one severity, we route per-speaker
    # For simplicity: route based on the first sample's severity
    # (per-speaker eval means all samples in window share same speaker/severity)
    batch_cats = [test_cats_filtered[j] for j in range(i, end)]
    sev_int    = TORGO_SEV_TO_INT[batch_cats[0]]
    model.set_active_severity(sev_int)

    batch = data_collator([
        {"input_values": test_p[j]["input_values"],
         "labels":       test_p[j]["labels"]}
        for j in range(i, end)
    ])

    input_values   = batch["input_values"].to(device)
    attention_mask = batch.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    with torch.no_grad():
        outputs = model(
            input_values=input_values,
            attention_mask=attention_mask,
        )

    pred_ids  = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
    pred_str  = decode_ctc(pred_ids, processor.tokenizer)

    label_ids = batch["labels"].numpy()
    label_ids = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    all_preds.extend([p.strip() for p in pred_str])
    all_labels.extend([l.strip() for l in label_str])
    all_cats.extend(batch_cats)

print(f"Decoded {len(all_preds)} TORGO test utterances.")

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Decoded 1786 TORGO test utterances.


In [21]:
# Overall metrics
eval_preds = [p if p else "⟨empty⟩" for p in all_preds]

overall_wer = wer_metric.compute(predictions=eval_preds, references=all_labels)
overall_cer = cer_metric.compute(predictions=eval_preds, references=all_labels)

print("=" * 55)
print("P1 — Severity-Stratified LoRA — TORGO TEST RESULTS")
print("=" * 55)
print(f"Overall WER: {overall_wer:.4f}  ({overall_wer*100:.2f}%)")
print(f"Overall CER: {overall_cer:.4f}  ({overall_cer*100:.2f}%)")

# Per-severity
results_df = pd.DataFrame({
    "prediction": all_preds,
    "reference":  all_labels,
    "Category":   all_cats,
})

print("\nPer-severity results:")
print("-" * 55)
for cat in ["control", "mild", "moderate", "severe"]:
    sub = results_df[results_df["Category"] == cat]
    sub = sub[sub["reference"].str.strip().str.len() > 0]
    if len(sub) == 0:
        print(f"{cat:15s}: no samples")
        continue
    preds = [p if p else "⟨empty⟩" for p in sub["prediction"].tolist()]
    cat_wer = wer_metric.compute(predictions=preds, references=sub["reference"].tolist())
    cat_cer = cer_metric.compute(predictions=preds, references=sub["reference"].tolist())
    print(f"{cat:15s}: WER={cat_wer*100:.2f}%  CER={cat_cer*100:.2f}%  (n={len(sub)})")

P1 — Severity-Stratified LoRA — TORGO TEST RESULTS
Overall WER: 0.3403  (34.03%)
Overall CER: 0.1674  (16.74%)

Per-severity results:
-------------------------------------------------------
control        : WER=10.44%  CER=4.00%  (n=302)
mild           : WER=13.07%  CER=4.29%  (n=673)
moderate       : WER=62.84%  CER=33.08%  (n=575)
severe         : WER=60.46%  CER=34.00%  (n=236)


In [22]:
# Sample predictions
print("\nSample predictions:")
print("-" * 70)
for i in range(min(15, len(all_preds))):
    print(f"[{all_cats[i]:10s}] REF : {all_labels[i]}")
    print(f"           PRED: {all_preds[i]}")
    print()

# Save results
results_df.to_csv("/kaggle/working/p1_torgo_test_results.csv", index=False)

summary = {
    "model": "P1_LoRA_Severity",
    "backbone": "B2 (frozen)",
    "lora_rank": 8,
    "lora_alpha": 16,
    "severity_loss": "EMD ordinal",
    "sev_loss_weight": 0.1,
    "validation_set": "UASpeech stratified",
    "overall_wer": round(overall_wer, 4),
    "overall_cer": round(overall_cer, 4),
    "test_speakers": ["F01", "F04", "FC01", "M05"],
}
with open("/kaggle/working/p1_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Results saved.")


Sample predictions:
----------------------------------------------------------------------
[control   ] REF : alpha
           PRED: alpha

[control   ] REF : the
           PRED: the

[control   ] REF : except in the winter when the oze or snow or ice prevents
           PRED: except in the winter when the ose or snow or ice prevents

[control   ] REF : raid
           PRED: raid

[control   ] REF : read
           PRED: lead

[control   ] REF : stuble
           PRED: stuble

[control   ] REF : ate
           PRED: ahte

[control   ] REF : store
           PRED: store

[control   ] REF : sip
           PRED: sip

[control   ] REF : wish
           PRED: wish

[control   ] REF : slay
           PRED: slay

[control   ] REF : sigh
           PRED: sigh

[control   ] REF : jaged
           PRED: jaged

[control   ] REF : up
           PRED: up

[control   ] REF : chair
           PRED: chair

Results saved.


In [23]:
# Zip for download
!zip -r /kaggle/working/p1_lora_severity_final.zip /kaggle/working/p1_lora_severity_final/
print("Zip ready.")

  adding: kaggle/working/p1_lora_severity_final/ (stored 0%)
  adding: kaggle/working/p1_lora_severity_final/vocab.json (deflated 56%)
  adding: kaggle/working/p1_lora_severity_final/lm_head.pt (deflated 8%)
  adding: kaggle/working/p1_lora_severity_final/added_tokens.json (deflated 20%)
  adding: kaggle/working/p1_lora_severity_final/severity_config.json (deflated 52%)
  adding: kaggle/working/p1_lora_severity_final/processor_config.json (deflated 44%)
  adding: kaggle/working/p1_lora_severity_final/severity_classifier.pt (deflated 8%)
  adding: kaggle/working/p1_lora_severity_final/model_state.pt (deflated 9%)
  adding: kaggle/working/p1_lora_severity_final/tokenizer_config.json (deflated 73%)
Zip ready.
